<a href="https://colab.research.google.com/github/krimits/hotel-review-nlp/blob/main/BILSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Κλωνοποίηση του αποθετηρίου σας από το GitHub
!git clone https://github.com/krimits/hotel-review-nlp
%cd hotel-review-nlp/hotel-review-nlp

# 2. Εγκατάσταση απαραίτητων πακέτων
!pip install wandb scikit-learn pandas torch

# 3. Σύνδεση στο Weights & Biases (Θα σας ζητηθεί το API Key)
import wandb
wandb.login()


Cloning into 'hotel-review-nlp'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 89 (delta 2), reused 89 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 319.18 KiB | 15.20 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/hotel-review-nlp/hotel-review-nlp


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krimits (krimits-aueb-students-investment-finance-club) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [12]:
yaml_content = """
seed: 100

data:
  processed_dir: data/processed
  max_tokens: 220
  min_freq: 2
  glove: null

model:
  hidden_dim: 128
  num_layers: 1
  bidirectional: true
  dropout: 0.3
  pooling: last

train:
  batch_size: 128
  epochs: 6
  embedding_dim: 128
  lr: 0.001
  weight_decay: 0.0001
  grad_clip: 5.0
  patience: 2

output:
  model_dir: runs/bilstm
"""

with open("configs/bilstm.yaml", "w") as f:
    f.write(yaml_content.strip())
print("Το αρχείο configs/bilstm.yaml ενημερώθηκε επιτυχώς!")


Το αρχείο configs/bilstm.yaml ενημερώθηκε επιτυχώς!


In [13]:
import os
import time
import json
import argparse
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import wandb

# Σημείωση: Εισάγουμε τα modules από το repo σας, προσθέτοντας το src στο path
import sys
sys.path.append("src")

from reviewnlp.data.dataset import TextVocab, build_bilstm_datasets, collate
from reviewnlp.evaluation.metrics import binary_metrics
from reviewnlp.utils.seed import load_config, set_seed

# --- Αρχιτεκτονική Μοντέλου ---
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int, num_layers: int = 1, dropout: float = 0.3, bidirectional: bool = True, pooling: str = "last", pad_idx: int = 0):
        super().__init__()
        self.pooling = pooling
        self.bidirectional = bidirectional
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0)
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(out_dim, 2)
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        with torch.no_grad():
            self.embedding.weight[pad_idx].zero_()

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.dropout(self.embedding(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, (h_n, _) = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        if self.pooling == "last":
            feats = torch.cat([h_n[-2], h_n[-1]], dim=-1) if self.bidirectional else h_n[-1]
        elif self.pooling == "max":
            feats = out.masked_fill(input_ids.unsqueeze(-1) == self.pad_idx, -1e4).max(dim=1).values
        else:
            mask = input_ids.ne(self.pad_idx).unsqueeze(-1).float()
            feats = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.fc(self.dropout(feats))

# --- Συνάρτηση Αξιολόγησης (Διορθωμένη) ---
def _evaluate(model: BiLSTMClassifier, loader: DataLoader, device: torch.device) -> tuple[float, dict, torch.Tensor]:
    model.eval()
    logits_all, preds, golds = [], [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"].to(device), batch["lengths"].to(device))
            logits_all.append(logits.float().cpu())
            preds += logits.argmax(dim=-1).cpu().tolist()
            golds += batch["labels"].tolist()
    acc = sum(p == g for p, g in zip(preds, golds, strict=False)) / max(1, len(golds))
    # ΔΙΟΡΘΩΣΗ: Αλλαγή από ("negative", "positive") σε (0, 1) λόγω αριθμητικών δεδομένων
    metrics = binary_metrics(golds, preds, label_names=(0, 1))
    return acc, metrics, torch.cat(logits_all)





# --- Κύρια Συνάρτηση Εκπαίδευσης με W&B ---
def train_bilstm(config_path: str) -> dict:
    cfg = load_config(config_path)
    set_seed(cfg["seed"])

    # Επιλογή Device (Αν έχετε ενεργοποιήσει GPU στο Colab, θα γράψει 'cuda')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Αρχικοποίηση του Weights & Biases Run
    wandb.init(
        entity="krimits-aueb-students-investment-finance-club",
        project="hotel-review-nlp",
        config=cfg,
        name="bilstm-weighted-ce-seed100",
        job_type="train",
        tags=["weighted-loss", "final-candidate"],
        reinit="finish_previous",
        save_code=True,
    )

    m, t, d = cfg["model"], cfg["train"], cfg["data"]
    train_ds, dev_ds, test_ds, vocab = build_bilstm_datasets(
        d["processed_dir"],
        d["max_tokens"],
        d.get("min_freq", 2),
    )

    class_counts, class_weights = balanced_class_weights(train_ds)

    print("Train class counts:", class_counts.tolist())
    print("Class weights:", class_weights.tolist())

    wandb.config.update({
        "loss_type": "class_weighted_cross_entropy",
        "train_class_counts": class_counts.tolist(),
        "class_weights": class_weights.tolist(),
    })

    print(
        f"vocab={len(vocab):,} train={len(train_ds):,} "
        f"dev={len(dev_ds):,} test={len(test_ds):,} device={device}"
    )
    print(f"vocab={len(vocab):,} train={len(train_ds):,} dev={len(dev_ds):,} test={len(test_ds):,} device={device}")

    loader_kwargs = dict(batch_size=t["batch_size"], collate_fn=collate, num_workers=2)
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    dev_loader = DataLoader(dev_ds, batch_size=256, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=256, collate_fn=collate)

    model = BiLSTMClassifier(
        vocab_size=len(vocab),
        embedding_dim=t["embedding_dim"],
        hidden_dim=m["hidden_dim"],
        num_layers=m["num_layers"],
        dropout=m["dropout"],
        bidirectional=m["bidirectional"],
        pooling=m["pooling"],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=t["lr"], weight_decay=t["weight_decay"])
    total_steps = len(train_loader) * t["epochs"]
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=t["lr"], total_steps=total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, t["epochs"] + 1):
        model.train()
        running, n_batches, t0 = 0.0, 0, time.perf_counter()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch["input_ids"].to(device), batch["lengths"].to(device))
            loss = criterion(logits, batch["labels"].to(device))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), t["grad_clip"])
            optimizer.step()
            scheduler.step()
            running += loss.item()
            n_batches += 1

        dev_acc, dev_metrics, _ = _evaluate(model, dev_loader, device)
        epoch_loss = running / n_batches
        print(f"epoch {epoch:02d} | loss {epoch_loss:.4f} | dev acc {dev_acc:.4f} | dev macro-F1 {dev_metrics['macro_f1']:.4f} | {time.perf_counter() - t0:.1f}s")

        # Live καταγραφή στο Weights & Biases
        wandb.log({
            "epoch": epoch,
            "train_loss": epoch_loss,
            "dev_accuracy": dev_acc,
            "dev_macro_f1": dev_metrics['macro_f1']
        })

        if dev_metrics["macro_f1"] > best_f1:
            best_f1, bad_epochs = dev_metrics["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= t["patience"]:
                print(f"early stop (no dev improvement for {t['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    test_acc, test_metrics, test_logits = _evaluate(model, test_loader, device)
    print(f"\nTEST RESULTS: acc={test_acc:.4f} macro-F1={test_metrics['macro_f1']:.4f}")

    # Καταγραφή των τελικών αποτελεσμάτων στο W&B
    test_log = flat_metrics("test", test_acc, test_metrics)
    test_log["best_dev_macro_f1"] = float(best_f1)
    test_log["class_0_weight"] = float(class_weights[0])
    test_log["class_1_weight"] = float(class_weights[1])
    wandb.log(test_log)
    out_dir = cfg["output"]["model_dir"]
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(out_dir, "bilstm.pt"))

    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump({"test": test_metrics, "best_dev_macro_f1": best_f1}, f, indent=2)

    wandb.finish()
    return test_metrics


In [14]:
def balanced_class_weights(train_ds):
    loader = DataLoader(
        train_ds,
        batch_size=1024,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
    )

    counts = torch.zeros(2, dtype=torch.long)

    for batch in loader:
        labels = batch["labels"].long()
        counts += torch.bincount(labels, minlength=2)

    weights = counts.sum() / (2 * counts.float().clamp_min(1))
    return counts, weights


def flat_metrics(prefix, accuracy, metrics):
    result = {
        f"{prefix}_accuracy": float(accuracy),
        f"{prefix}_macro_f1": float(metrics["macro_f1"]),
        f"{prefix}_macro_precision": float(metrics["macro_precision"]),
        f"{prefix}_macro_recall": float(metrics["macro_recall"]),
        f"{prefix}_weighted_f1": float(metrics["weighted_f1"]),
    }

    for label, values in metrics["per_class"].items():
        result[f"{prefix}_class_{label}_precision"] = float(values["precision"])
        result[f"{prefix}_class_{label}_recall"] = float(values["recall"])
        result[f"{prefix}_class_{label}_f1"] = float(values["f1"])
        result[f"{prefix}_class_{label}_support"] = int(values["support"])

    return result

In [15]:
# 1. Δημιουργία των απαραίτητων φακέλων
!mkdir -p data/raw data/processed

# 2. Εκτέλεση του preprocess script για να δημιουργηθούν τα αρχεία .parquet
!python -m reviewnlp.data.preprocess --config configs/baselines.yaml


/usr/bin/python3: Error while finding module specification for 'reviewnlp.data.preprocess' (ModuleNotFoundError: No module named 'reviewnlp')


In [16]:
from pathlib import Path
print("cwd:", Path.cwd())
for f in ("train.parquet", "dev.parquet", "test.parquet"):
    print(f, (Path("data/processed") / f).resolve(), (Path("data/processed") / f).exists())

cwd: /content/hotel-review-nlp/hotel-review-nlp
train.parquet /content/hotel-review-nlp/hotel-review-nlp/data/processed/train.parquet True
dev.parquet /content/hotel-review-nlp/hotel-review-nlp/data/processed/dev.parquet True
test.parquet /content/hotel-review-nlp/hotel-review-nlp/data/processed/test.parquet True


In [17]:
import os, wandb
from pathlib import Path

wandb.finish(exit_code=1)
os.chdir("/content/hotel-review-nlp/hotel-review-nlp")

files = [Path("data/processed") / f"{s}.parquet" for s in ("train", "dev", "test")]
assert all(f.exists() for f in files), [(str(f.resolve()), f.exists()) for f in files]

train_bilstm("configs/bilstm.yaml")

Train class counts: [26232, 92758]
Class weights: [2.268031358718872, 0.6414002180099487]
vocab=15,138 train=118,990 dev=14,872 test=13,278 device=cuda
vocab=15,138 train=118,990 dev=14,872 test=13,278 device=cuda
epoch 01 | loss 0.3614 | dev acc 0.9543 | dev macro-F1 0.9356 | 23.4s
epoch 02 | loss 0.1418 | dev acc 0.9538 | dev macro-F1 0.9358 | 23.5s
epoch 03 | loss 0.1188 | dev acc 0.9534 | dev macro-F1 0.9355 | 24.4s
epoch 04 | loss 0.1045 | dev acc 0.9607 | dev macro-F1 0.9446 | 23.9s
epoch 05 | loss 0.0916 | dev acc 0.9609 | dev macro-F1 0.9450 | 23.0s
epoch 06 | loss 0.0841 | dev acc 0.9613 | dev macro-F1 0.9452 | 23.0s

TEST RESULTS: acc=0.9614 macro-F1=0.9492


best_dev_macro_f1,▁
class_0_weight,▁
class_1_weight,▁
dev_accuracy,▂▁▁███
dev_macro_f1,▁▁▁███
epoch,▁▂▄▅▇█
test_accuracy,▁
test_class_0_f1,▁
test_class_0_precision,▁
test_class_0_recall,▁
+10,...


{'accuracy': 0.9614,
 'macro_f1': 0.9492,
 'macro_precision': 0.9401,
 'macro_recall': 0.9594,
 'weighted_f1': 0.9618,
 'per_class': {0: {'precision': 0.8951,
   'recall': 0.9555,
   'f1': 0.9243,
   'support': 3278},
  1: {'precision': 0.9851, 'recall': 0.9633, 'f1': 0.9741, 'support': 10000}},
 'confusion_matrix': [[3132, 146], [367, 9633]]}